# CRANE Kaggle GPU Server v2
Ollama on 2x T4 (~30 GB VRAM) exposed through a Cloudflare tunnel for OpenCode.

- **Agent model:** `qwen3-coder:30b` (tool calling, ~19 GB, spans both GPUs, 32k context)
- **Uncensored chat model:** `dolphin3:8b` (no tool calling)
- Models live in `/tmp` (the 20 GB `/kaggle/working` limit would break the run)

Settings: Accelerator = GPU T4 x2, Internet = On. Run all cells. Cell 4 prints the tunnel URL and saves it to `/kaggle/working/tunnel_url.txt`.

In [ ]:
# Cell 1 — system deps + Ollama
import subprocess, os
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd pciutils', shell=True, check=True)
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, capture_output=True, text=True)
assert os.path.exists('/usr/local/bin/ollama'), 'Ollama install FAILED:\n' + r.stdout[-800:] + r.stderr[-800:]
print(subprocess.run('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader; free -g | head -2; df -h /tmp | tail -1', shell=True, capture_output=True, text=True).stdout)
print('Ollama installed')

In [ ]:
# Cell 2 — tuned Ollama server
import subprocess, time, os, urllib.request
MODELS_DIR = '/tmp/ollama_models'
os.makedirs(MODELS_DIR, exist_ok=True)
ENV = {**os.environ,
  'OLLAMA_HOST': '0.0.0.0:11434',
  'OLLAMA_MODELS': MODELS_DIR,
  'OLLAMA_FLASH_ATTENTION': '1',
  'OLLAMA_KV_CACHE_TYPE': 'q8_0',     # halves KV-cache VRAM, room for long context
  'OLLAMA_KEEP_ALIVE': '-1',          # never unload
  'OLLAMA_MAX_LOADED_MODELS': '2',    # agent + chat model resident together
  'OLLAMA_NUM_PARALLEL': '2',
  'OLLAMA_SCHED_SPREAD': '1',         # balance layers across both T4s
}
def start_ollama():
    p = subprocess.Popen(['/usr/local/bin/ollama','serve'], env=ENV,
                         stdout=open('/tmp/ollama.log','w'), stderr=subprocess.STDOUT)
    for _ in range(30):
        try: urllib.request.urlopen('http://localhost:11434'); return p
        except Exception: time.sleep(1)
    raise RuntimeError('Ollama did not start:\n' + open('/tmp/ollama.log').read()[-1500:])
ollama_proc = start_ollama()
print('Ollama UP')

In [ ]:
# Cell 3 — pull models and build 32k-context variants (OpenCode needs a large num_ctx for tool use)
import subprocess
def sh(c): 
    r = subprocess.run(c, shell=True, env=ENV, capture_output=True, text=True); print(r.stdout[-400:] or r.stderr[-400:]); return r.returncode

AGENT = 'qwen3-coder:30b'
CHAT  = 'dolphin3:8b'
for m in (AGENT, CHAT):
    assert sh(f'/usr/local/bin/ollama pull {m}') == 0, f'pull {m} failed'

open('/tmp/Modelfile.agent','w').write(f'''FROM {AGENT}
PARAMETER num_ctx 32768
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER repeat_penalty 1.05
''')
open('/tmp/Modelfile.chat','w').write(f'''FROM {CHAT}
PARAMETER num_ctx 32768
PARAMETER temperature 0.8
''')
sh('/usr/local/bin/ollama create crane-agent -f /tmp/Modelfile.agent')
sh('/usr/local/bin/ollama create crane-chat -f /tmp/Modelfile.chat')
# preload so the first request is fast
for m in ('crane-agent','crane-chat'):
    sh(f"curl -s localhost:11434/api/generate -d '{{\"model\":\"{m}\",\"keep_alive\":-1}}'")
print(subprocess.run('/usr/local/bin/ollama ps; nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader', shell=True, env=ENV, capture_output=True, text=True).stdout)

In [ ]:
# Cell 4 — Cloudflare tunnel, prints + saves the URL
import subprocess, threading, re, time
subprocess.run('curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
tunnel = {'url': None, 'proc': None}
def run_tunnel():
    tunnel['url'] = None
    p = subprocess.Popen(['/usr/local/bin/cloudflared','tunnel','--url','http://localhost:11434'],
                         stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True)
    tunnel['proc'] = p
    for line in p.stderr:
        m = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if m and not tunnel['url']:
            tunnel['url'] = m.group()
            open('/kaggle/working/tunnel_url.txt','w').write(tunnel['url'])
threading.Thread(target=run_tunnel, daemon=True).start()
for _ in range(40):
    if tunnel['url']: break
    time.sleep(1)
print('=' * 60); print('TUNNEL URL:', tunnel['url']); print('=' * 60)
print('On your laptop run:  bash /mnt/elana/ai_apps/crane/scripts/kaggle_set_url.sh', tunnel['url'])

In [ ]:
# Cell 5 — watchdog: restarts Ollama / tunnel if either dies (run last, keeps session alive)
import time, urllib.request
print('Watchdog running. Interrupt kernel to stop.')
while True:
    time.sleep(60)
    try: urllib.request.urlopen('http://localhost:11434', timeout=5)
    except Exception:
        print('\nOllama died - restarting'); ollama_proc = start_ollama()
    if tunnel['proc'] is None or tunnel['proc'].poll() is not None:
        print('\nTunnel died - restarting'); threading.Thread(target=run_tunnel, daemon=True).start()
        time.sleep(15); print('NEW URL:', tunnel['url'])
    print('.', end='', flush=True)